In [ ]:
!pip install sentencepiece

In [ ]:
import collections
import os
import pathlib
import re
import string
import sys
import tempfile
import time
import requests

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import tensorflow_datasets as tfds
import tensorflow_text as text
import tensorflow as tf
from tensorflow_text.tools.wordpiece_vocab import bert_vocab_from_dataset as bert_vocab

In [ ]:
def download_uniprot_human_proteome():

    stream_url = "https://rest.uniprot.org/uniprotkb/stream"
    params = {
        "query": "reviewed:true AND organism_id:9606",
        "format": "tsv",
        "fields": "accession,id,gene_primary,organism_name,protein_name,ec,go_f,go_p,go_c,sequence"
    }

    try:
        response = requests.get(stream_url, params=params)

        if response.status_code != 200:
            raise Exception(f"Stream request failed: {response.status_code} - {response.text[:200]}")

        content = response.text.strip()
        lines = content.split('\n')

        return lines

    except requests.exceptions.RequestException as e:
        print(f"Network error: {e}")
        return []

def save_tsv_data(data, filename="uniprot_human_swissprot.tsv"):
    """
    Save the dataset to a TSV file with proper formatting
    """
    if not data:
        print("No data to save")

    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write("\n".join(data))

    except IOError as e:
        print(f"File save error: {e}")

data = download_uniprot_human_proteome()
save_tsv_data(data)

In [ ]:
def parse_data(tsv_file, protein_path='protein.csv', output_path='output.csv'):
    df = pd.read_csv(tsv_file, sep="\t")
    protein_df = pd.DataFrame(columns=["Sequence"])

    output_df = pd.DataFrame(columns=["Outputs"])

    df.dropna(inplace = True)

    s = []

    for index, row in df.iterrows():
        sequence = row['Sequence']

        organism = row['Organism']
        protein_name = row["Protein names"]
        molecular_function = row['Gene Ontology (molecular function)']
        biological_process = row['Gene Ontology (biological process)']
        cellular_component = row['Gene Ontology (cellular component)']

        combined = " ".join([
            f'Organism: {str(organism)}',
            f'Protein_name: {str(protein_name)}',
            f'Molecular Function: {str(molecular_function)}',
            f'Biological Process: {str(biological_process)}',
            f'Cellular Component: {str(cellular_component)}'
        ])

        protein_df.loc[len(protein_df)] = sequence

        output_df.loc[len(output_df)] = combined

        s.append(sequence)

    protein_df.to_csv(protein_path, index=False)

    output_df["Outputs"].to_csv("outputs.txt", header=False, index=False)

    return protein_path, output_path, s

In [ ]:
protein_path, output_path, s = parse_data('/kaggle/working/uniprot_human_swissprot.tsv')

In [ ]:
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
    '--input=/kaggle/working/outputs.txt '
    '--model_prefix=biotext '
    '--vocab_size=900 '
    '--character_coverage=1.0 '
    '--model_type=bpe'
)

In [ ]:
def kmer_tokenize(sequence, k=3):
    """
    Split a protein sequence into overlapping k-mers.
    
    Args:
        sequence (str): Protein sequence (e.g. "MKTLLS").
        k (int): Size of k-mer (default=3).
    
    Returns:
        list: List of k-mer tokens.
    """
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

In [ ]:
from collections import Counter

def build_kmer_vocab(sequences, k=3, min_freq=1):
    """
    Build a vocabulary of k-mers from protein sequences.
    
    Args:
        sequences (list of str): Protein sequences.
        k (int): Size of k-mer.
        min_freq (int): Minimum frequency for inclusion in vocab.
    
    Returns:
        dict: {kmer: index}
    """
    counter = Counter()
    for seq in sequences:
        counter.update(kmer_tokenize(seq, k))

    vocab = {kmer: idx for idx, (kmer, count) in enumerate(counter.items(), start=2) if count >= min_freq}
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1

    return vocab

In [ ]:
vocab = build_kmer_vocab(s)

In [ ]:
def encode_sequence(seq, vocab, k=3):
    tokens = kmer_tokenize(seq, k)
    return [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]

In [ ]:
import json

with open("vocab_protein.json", "w") as file:
    json.dump(vocab, file, indent=4)

In [ ]:
def parse_data_dataset(tsv_file, output_path='dataset.csv'):
    df = pd.read_csv(tsv_file, sep="\t")
    full_df = pd.DataFrame(columns=["Sequence", "Outputs"])

    df.dropna(inplace = True)

    for index, row in df.iterrows():
        sequence = row['Sequence']

        organism = row['Organism']
        protein_name = row["Protein names"]
        molecular_function = row['Gene Ontology (molecular function)']
        biological_process = row['Gene Ontology (biological process)']
        cellular_component = row['Gene Ontology (cellular component)']

        combined = " ".join([
            f'Organism: {str(organism)}',
            f'Protein_name: {str(protein_name)}',
            f'Molecular Function: {str(molecular_function)}',
            f'Biological Process: {str(biological_process)}',
            f'Cellular Component: {str(cellular_component)}'
        ])

        full_df.loc[len(full_df)] = [sequence, combined]

    full_df.to_csv(output_path, index=False)

    return output_path

In [ ]:
output_path = parse_data_dataset('/kaggle/working/uniprot_human_swissprot.tsv')